# Run the linker analysis for GT

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

# CryoCAT
from cryocat import cryomap
from cryocat import cryomotl
from cryocat import ribana as ra
#from cryocat import motl_conversions as mc


#DNA Linker
from dna_linker import dna_linkers as dnal
from dna_linker import create_graph as cgraph
from dna_linker import main_dna_linkers as main
from dna_linker import utils_motlFiles as util_motl

# Muliprocessing
from multiprocessing import Pool

# Define paths

In [2]:
#[13356, 2601, 13363, 13370, 13379, 38407]
emd=2601 #13356 #38407

path='./'
path_mask='./inputs_noisy/'
#path_output='/Users/sergiocruz/Desktop/VisualProteomics/3DTM/T-cells/Resting/Nucleosome_classification/git_repo/linker_prediction/Notebook/Nova_interpol_TM_019/out_tracing/'
path_output='./out_tracing/'

entry='Threshold_ref_entrymask_r2_resamp_righthand.mrc'
exit='Threshold_ref_exitmask_r2_resamp_righthand.mrc'
origin_entry='Threshold_ref_Origin_entrymask_r2_resamp_righthand.mrc'
origin_exit='Threshold_ref_Origin_exitmask_r2_resamp_righthand.mrc'

## Change accorging to the gt
motl_name=f'EMD{emd}_noise_angle_1deg.em'#f'EMD{emd}_noise_25.em'#'motl_EMD{emd}_STA_tmpl.em'
#motl_name='motl_EMD_STA_tmpl.em'

#path_mask='./out_tracing/'
masks_path='./inputs/'
output_path_cluster=f'./outputs/outputs_EMD{emd}/clusters_20nm/'

#path_mask='./out_tracing/'
#output_path='./outputs/clusters_20nm/'
output_path_linker=f'./outputs/outputs_EMD{emd}//A_linkers_20nm/'
output_path_dictionary=f'./outputs/outputs_EMD{emd}/A_Connections_dictionary_20nm/'

# Params

In [3]:
tracing_distance=200. # Extenrnal user parameter - Distance
pixel_size = 1. # Models are at in the right scale
bin=1.
max_distance=tracing_distance/(pixel_size*bin)

name_traced=f'EMD{emd}_tr{tracing_distance}nm.em'
motl_trace_input=path_output+name_traced

# PART 1:Shift of the center of the particles to the entry and exit of the nucleosome particle
*Required for the tracing*

In [4]:
motl_all=cryomotl.EmMotl(path_mask+motl_name)
mask_entry=path_mask+entry
mask_exit=path_mask+exit
sh_motl_entry = cryomotl.Motl.recenter_to_subparticle(motl_all.df, mask_entry)
sh_motl_exit = cryomotl.Motl.recenter_to_subparticle(motl_all.df, mask_exit)

output_entry=path_output+'entry_b1_Trest_run_data_righthanded.em'
output_exit=path_output+'exit_b1_Trest_run_data_righthanded.em'

# Write out the shifted centers to the entry 
sh_motl_entry.write_out(output_entry)

# Write out the shifted centers to the exit 
sh_motl_exit.write_out(output_exit)

In [5]:
%%time

traced_motl=ra.trace_chains(output_entry,output_exit,max_distance=max_distance, min_distance=0)
traced_motl.df.sort_values(['tomo_id','object_id', 'geom2'], inplace=True)

entry_motl_traced=path_output+f'entry_b1_Trest_run_data_righthanded_tr{tracing_distance}nm.em'
exit_motl_traced=path_output+f'exit_b1_Trest_run_data_righthanded_tr{tracing_distance}nm.em'
output_motl_traced=motl_trace_input

################################################################################################
################################################################################################
ra.add_occupancy(traced_motl)
ra.add_traced_info(traced_motl, output_entry, entry_motl_traced)
ra.add_traced_info(traced_motl, output_exit, exit_motl_traced)
ra.add_traced_info(traced_motl, motl_all, 
                   output_motl_traced)


CPU times: user 25.6 ms, sys: 2.65 ms, total: 28.3 ms
Wall time: 29.4 ms


# PART 2: Tracing

## Create lists per tomo per cluster

In [6]:
%%time
util_motl.create_motllists_perTomo_perCluster(motl_trace_input=motl_trace_input,
                                    output_path=output_path_cluster)


The file:  ./outputs/outputs_EMD2601/clusters_20nm/motl_tomo0.0_cluster12.0.em has been saved!
CPU times: user 1.82 ms, sys: 1.07 ms, total: 2.9 ms
Wall time: 3.19 ms


## Create lists for the entry, entry2, exit and exit2

### The following creates all the lists (entry, entry2, exit, exit2) for all tomogram/cluster combinations

In [7]:
%%time
## Load the initial trace to loop over the tomograms and the clusters
motl_trace=cryomotl.EmMotl(input_motl=motl_trace_input)
tomograms=motl_trace.df['tomo_id'].unique()
for tomo_id in tomograms:
    df_motl_tomo=motl_trace.df[motl_trace.df['tomo_id']==tomo_id]
    clusters=df_motl_tomo['geom1'].unique()
    for cluster in clusters:
        ## This will create the 4-lists per tomo_id per cluster
        output_paths=util_motl.create_motiflists_for_entry_exit(output_path=output_path_cluster, tomo_id=tomo_id, cluster=cluster,
                           masks_path=path_mask, mask_origin_entry=origin_entry, 
                           mask_entry=entry,mask_origin_exit=origin_exit, mask_exit=exit,
                                prefix='All_')

CPU times: user 521 ms, sys: 58.1 ms, total: 579 ms
Wall time: 596 ms


# Create complete EmMotl for the entry and exit of all the particles

In [8]:
%%time
## Load the initial tra\\ce to loop over the tomograms and the clusters
motl_trace=cryomotl.EmMotl(input_motl=motl_trace_input)
output_paths=util_motl.create_shifted_motiflists_along_linker(output_path=output_path_cluster, path_motl=motl_trace_input, 
                           masks_path=path_mask, mask_origin_entry=origin_entry, 
                          mask_entry=entry,mask_origin_exit=origin_exit, mask_exit=exit,
                                prefix='All_')

CPU times: user 510 ms, sys: 54.2 ms, total: 564 ms
Wall time: 582 ms


# PART 3: Determine connectivity 

In [9]:
%%time
if __name__ == '__main__':
    ## Test for the distances
    motl_trace=cryomotl.EmMotl(input_motl=motl_trace_input)
    tomograms=motl_trace.df['tomo_id'].unique()
    paths_conn=[]
    largest_components=[]
    packed_paths = []
    for tomo_id in tomograms:
            print(f'Tomogram: {tomo_id}')
            df_motl_tomo=motl_trace.df[motl_trace.df['tomo_id']==tomo_id]
            clusters=df_motl_tomo['geom1'].unique()

            
            for cluster in clusters:
                #if cluster >2:#  and cluster>=2: # and cluster<=11:
                    packed_paths.append([tomo_id, cluster, output_path_cluster, output_path_linker, 
                                  output_path_dictionary, dnal.lo])
            
            # Set the number of worker processes
            num_processes = min(10, len(packed_paths))  # Limit to 4 processes or less if fewer tasks

            #with Pool(len(packed_paths)) as p:
            with Pool(num_processes) as p:
                p.map(main.main_function, packed_paths)

Tomogram: 0.0
Tomogram: 0.0, and cluster 12.0
Figure(640x480)
(12, 12, 4)
Largest connected component: {2, 7}
Size of largest connected component: 2
{2, 7}
Figure(640x480)
CPU times: user 12 ms, sys: 31.1 ms, total: 43.1 ms
Wall time: 4.15 s
